In [ ]:
# 종목별 LSTM 모델을 생성 및 저장하는 파일 (윈도우, tf 2.18, h5)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import ta

In [ ]:
# 완료된, 실패한 종목 관리 파일
completed_file = '../data/completed_symbols_h5.txt'
failed_file = '../data/failed_symbols_h5.txt'

# 기존 파일 읽기
completed_symbols = set()
if os.path.exists(completed_file):
    with open(completed_file, 'r') as f:
        completed_symbols = set(f.read().splitlines())

failed_symbols = set()
if os.path.exists(failed_file):
    with open(failed_file, 'r') as f:
        failed_symbols = set(f.read().splitlines())

def load_progress():
    if os.path.exists(completed_file):
        with open(completed_file, 'r') as f:
            done_symbols = set([line.strip() for line in f if line.strip()])
    else:
        done_symbols = set()
    return done_symbols

In [ ]:
import joblib

# sMAPE 계산 함수
def smape(a, f):
    return 100 / len(a) * np.sum(2 * np.abs(f - a) / (np.abs(a) + np.abs(f)))

# 모델 저장 경로
MODEL_FOLDER = '../model/LSTM_MODEL_H5_WIN'
os.makedirs(MODEL_FOLDER, exist_ok=True)

# 종목별 평가 지표 저장용
results = []

# 전체 CSV 로딩
df_all = pd.read_csv('../data/csv/sp500_latest.csv')
symbols = df_all['Symbol'].unique()
done_symbols = load_progress()
print(f"[INFO] Symbols already processed (will skip): {len(done_symbols)}")

for symbol in symbols:
    
    if symbol in done_symbols:
        print(f"[SKIP] Already processed {symbol}")
        continue

    try:
        df = df_all[df_all['Symbol'] == symbol].copy()
        df = df.dropna()

        # 기술적 지표 추가
        df['MA20'] = ta.trend.sma_indicator(df['Close'], window=20)
        bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
        df['Upper'] = bb.bollinger_hband()
        df['Lower'] = bb.bollinger_lband()
        df['RSI'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()
        df.dropna(inplace=True)

        features = ['Open', 'High', 'Low', 'Close', 'Volume', 'MA20', 'Upper', 'Lower', 'RSI']
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(df[features])

        sequence_length = 50
        X, y = [], []
        for i in range(len(scaled_data) - sequence_length):
            X.append(scaled_data[i:i+sequence_length])
            y.append(scaled_data[i+sequence_length][features.index('Close')])
        X = np.array(X)
        y = np.array(y)

        # 7:2:1 split
        train_size = int(len(X) * 0.7)
        val_size = int(len(X) * 0.2)
        X_train, X_val, X_test = X[:train_size], X[train_size:train_size+val_size], X[train_size+val_size:]
        y_train, y_val, y_test = y[:train_size], y[train_size:train_size+val_size], y[train_size+val_size:]

        # 모델 구성
        model = Sequential([
            LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
            Dropout(0.3),
            LSTM(32),
            Dropout(0.3),
            Dense(1)
        ])
        model.compile(optimizer='adam', loss='mse')
        early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

        # 학습
        model.fit(X_train, y_train, epochs=50, batch_size=32,
                  validation_data=(X_val, y_val), callbacks=[early_stop], verbose=0)
        
        # 모델 저장
        # model.save(os.path.join(MODEL_FOLDER, symbol), save_format='tf') # TF 방식 
        model.save(os.path.join(MODEL_FOLDER, f'{symbol}.h5')) # H5 방식
        
        # 스케일러 저장
        # joblib.dump(scaler, '/home/danssa/proj_ua/shared/scaler60.pkl') # pkl 방식 
        joblib.dump(scaler, os.path.join(MODEL_FOLDER, f'{symbol}_scaler.joblib')) # joblib 방식
        
        
        # 성공시 기록
        with open(completed_file, 'a') as f:
            f.write(f"{symbol}\n")


        # 예측 및 역변환
        pred = model.predict(X_test)
        close_index = features.index('Close')
        y_full = np.zeros((len(y_test), len(features)))
        pred_full = np.zeros((len(pred), len(features)))
        y_full[:, close_index] = y_test
        pred_full[:, close_index] = pred.flatten()
        true_rescaled = scaler.inverse_transform(y_full)[:, close_index]
        pred_rescaled = scaler.inverse_transform(pred_full)[:, close_index]

        # 평가
        smape_val = smape(true_rescaled, pred_rescaled)
        mae_val = mean_absolute_error(true_rescaled, pred_rescaled)
        mse_val = mean_squared_error(true_rescaled, pred_rescaled)

        results.append({
            'Symbol': symbol,
            'sMAPE': smape_val,
            'MAE': mae_val,
            'MSE': mse_val
        })

        print(f"{symbol} 완료 - sMAPE: {smape_val:.2f}%, MAE: {mae_val:.2f}, MSE: {mse_val:.2f}")

    except Exception as e:
        print(f"{symbol} 실패: {e}")
        # 실패 기록
        with open(failed_file, 'a') as f:
            f.write(f"{symbol}\n")




# 결과 정리
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('sMAPE')

# sMAPE 구간화
bins = np.arange(0, 105, 5)
labels = [f'{i}~{i+5}%' for i in bins[:-1]]
results_df['sMAPE_Group'] = pd.cut(results_df['sMAPE'], bins=bins, labels=labels, right=False)

print("\n--- Top 10 종목 (sMAPE 낮은 순) ---")
print(results_df.head(10))

print("\n--- sMAPE 오차 범위별 종목 개수 (5% 단위) ---")
print(results_df['sMAPE_Group'].value_counts().sort_index())

print("\n--- 전체 지표 ---")
print(results_df)

print("\nGPU 상태:")
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("Available devices:", tf.config.list_physical_devices())

In [ ]:
# # 1~5%  : 72.5%
# # 5~10% : 20.8% 


--- Top 10 종목 (sMAPE 낮은 순) ---
  Symbol     sMAPE sMAPE_Group
0   GILD  1.076772        0~5%
1     MO  1.226468        0~5%
2    CPB  1.253405        0~5%
3    KMB  1.372075        0~5%
4   FOXA  1.563924        0~5%
5    FOX  1.567218        0~5%
6    LYB  1.599551        0~5%
7      K  1.638793        0~5%
8     PM  1.648253        0~5%
9      O  1.671763        0~5%

--- sMAPE 오차 범위별 종목 개수 (5% 단위) ---
sMAPE_Group
0~5%       359
5~10%      103
10~15%      24
15~20%       7
20~25%       0
25~30%       0
30~35%       1
35~40%       0
40~45%       0
45~50%       0
50~55%       0
55~60%       0
60~65%       0
65~70%       0
70~75%       0
75~80%       0
80~85%       0
85~90%       0
90~95%       0
95~100%      0
Name: count, dtype: int64
